In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/VideoRAG/VideoRAG/VideoRAG-algorithm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/VideoRAG/VideoRAG/VideoRAG-algorithm


In [2]:
# Core numerical and deep learning libraries
!pip install numpy==1.26.4
!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2
!pip install accelerate==0.30.1
!pip install bitsandbytes==0.43.1

# Video processing utilities
!pip install moviepy==1.0.3
!pip install git+https://github.com/facebookresearch/pytorchvideo.git@28fe037d212663c6a24f373b94cc5d478c8c1a1d
!pip install --no-deps git+https://github.com/facebookresearch/ImageBind.git@3fcf5c9039de97f6ff5528ee4a9dce903c5979b3

# Multi-modal and vision libraries
!pip install timm ftfy regex einops fvcore eva-decord==0.6.1 iopath matplotlib types-regex cartopy

# Audio processing and vector databases
!pip install ctranslate2==4.4.0 faster_whisper==1.0.3 neo4j hnswlib xxhash nano-vectordb

# Language models and utilities
!pip install transformers==4.37.1
!pip install tiktoken openai tenacity
!pip install ollama==0.5.3

  Using cached torch-2.1.2-cp311-cp311-manylinux1_x86_64.whl.metadata (25 kB)
  Using cached torchvision-0.16.2-cp311-cp311-manylinux1_x86_64.whl.metadata (6.6 kB)
  Using cached torchaudio-2.1.2-cp311-cp311-manylinux1_x86_64.whl.metadata (6.4 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 k

In [3]:
import os
!pip install nvidia-cudnn-cu12==8.9.2.26

import nvidia.cudnn
cudnn_dir = os.path.dirname(nvidia.cudnn.__file__)

# 3. Add that folder to the system's "look here for libraries" path
os.environ['LD_LIBRARY_PATH'] = f"{cudnn_dir}/lib:" + os.environ.get('LD_LIBRARY_PATH', '')

# 1. Copy all cuDNN 8 shared libraries directly into the OS's primary library folder
!cp -a /usr/local/lib/python3.11/dist-packages/nvidia/cudnn/lib/libcudnn*.so* /usr/lib/x86_64-linux-gnu/

# 2. Update the system's dynamic linker cache so it registers the newly pasted files immediately
!ldconfig



/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/l

# run once

In [ ]:
# !git lfs install
# !git lfs clone https://huggingface.co/openbmb/MiniCPM-V-2_6-int4
# !git lfs clone https://huggingface.co/Systran/faster-distil-whisper-large-v3

# !mkdir .checkpoints
# %cd .checkpoints
# !wget https://dl.fbaipublicfiles.com/imagebind/imagebind_huge.pth
# %cd ../

# set up ollama

In [4]:
import os
import subprocess
import time

!sudo apt-get update && sudo apt-get install -y zstd

# 1. Create a folder in your Drive for Ollama
drive_ollama_path = "/content/drive/MyDrive/VideoRAG/ollama_models"
os.makedirs(drive_ollama_path, exist_ok=True)

# 2. Tell Ollama to use this Drive folder
env = os.environ.copy()
env["OLLAMA_MODELS"] = drive_ollama_path

# 3. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 4. Start the server in the background (using your Drive path)
subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(3) # Give it a few seconds to boot up

# 5. Pull the VRAM-friendly models
# (This takes a few minutes the first time. In future sessions, it will be instant!)
!ollama pull nomic-embed-text
!ollama pull gemma2:latest
!ollama pull olmo2

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,475 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,954 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get

In [ ]:
# This will ask your local Ollama server to process a tiny prompt
!ollama run gemma2:latest "what is the capital of vietnam"

The capital of Vietnam is **Hanoi**. 




# code

In [ ]:
import os
import logging
import warnings
import multiprocessing

import nest_asyncio

warnings.filterwarnings("ignore")
logging.getLogger("httpx").setLevel(logging.WARNING)

# Please enter your openai key
#os.environ["OPENAI_API_KEY"] = ""

#from videorag._llm import openai_4o_mini_config
from videorag._llm import ollama_config
from videorag import VideoRAG, QueryParam

nest_asyncio.apply()


video_paths = [
    '/content/drive/MyDrive/VideoRAG/Videos/Sintel.mp4',
]
videorag = VideoRAG(llm=ollama_config, working_dir="/content/drive/MyDrive/VideoRAG/VideoRAG/VideoRAG-algorithm/videorag-workdir")
videorag.insert_video(video_path_list=video_paths)

Starting....


Speech Recognition Sintel: 100%|██████████| 30/30 [00:41<00:00,  1.39s/it]


[0.00s -> 1.00s]  ...in...
[1.00s -> 2.00s]  ...their...
[2.00s -> 4.00s]  ...more An...
[4.00s -> 5.00s]  ...with...
[5.00s -> 6.00s]  ...when...
[6.00s -> 7.00s]  ...the...



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Captioning Video Sintel: 100%|██████████| 30/30 [18:35<00:00, 37.19s/it]


Starting.. ainsert
Sequentially processing 9 chunks...
Sending Chunk 1/9 to Ollama...
Sending Chunk 2/9 to Ollama...
Sending Chunk 3/9 to Ollama...
Sending Chunk 4/9 to Ollama...
Sending Chunk 5/9 to Ollama...
Sending Chunk 6/9 to Ollama...
Sending Chunk 7/9 to Ollama...
Sending Chunk 8/9 to Ollama...
Sending Chunk 9/9 to Ollama...


In [ ]:
# import os
# import logging
# import warnings
# import multiprocessing

# import nest_asyncio
# nest_asyncio.apply()

# warnings.filterwarnings("ignore")
# logging.getLogger("httpx").setLevel(logging.WARNING)

# # Please enter your openai key
# os.environ["OPENAI_API_KEY"] = ""

# from videorag._llm import ollama_config
# from videorag import VideoRAG, QueryParam


# if __name__ == '__main__':
#     try:
#       multiprocessing.set_start_method('spawn')
#     except RuntimeError:
#       pass

#     query = 'Can you make a sumary of the video content'
#     param = QueryParam(mode="videorag")
#     # if param.wo_reference = False, VideoRAG will add reference to video clips in the response
#     param.wo_reference = True

#     videorag = videorag = VideoRAG(llm=ollama_config, working_dir=f"/content/drive/MyDrive/VideoRAG/VideoRAG/VideoRAG-algorithm/videorag-workdir")
#     videorag.load_caption_model(debug=False)
#     response = videorag.query(query=query, param=param)
#     print(response)